## Validating FinBERT

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from msa.utils.paths import get_processed_data_path

processed = get_processed_data_path()

df = pd.read_csv(
    processed / "gdelt_articles_with_sentiment.csv",
    parse_dates=["seendate"]
)

print(f"Loaded {len(df):,} articles")
print(f"Date range: {df['seendate'].min().date()} → {df['seendate'].max().date()}")

Loaded 7,224 articles
Date range: 2024-02-08 → 2026-03-04


In [25]:
MAG7 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA']

# v1: dictionary proxy — mirrors original 44% coverage
# NOTE: original file overwritten by FinBERT pipeline; baseline from test_finbert_sample.py output
v1 = df.copy()
v1["sentiment_present"] = v1["sentiment_label"] != "neutral"

# v2: FinBERT — every article has a score, no hard neutral filter
v2 = df.copy()
v2["sentiment_present"] = True

v1_mag7 = v1[v1["ticker"].isin(MAG7)].copy()
v2_mag7 = v2[v2["ticker"].isin(MAG7)].copy()

# Documented baselines from mag7_data_flow_analysis.md + test output
V1_SENTIMENT_RATE = 0.44
V1_DAILY_PAIRS    = 462
V1_DAYS_5PLUS     = 55

print(f"v1 sentiment rate: {v1['sentiment_present'].mean():.1%}")
print(f"v2 sentiment rate: {v2['sentiment_present'].mean():.1%}")

v1 sentiment rate: 43.3%
v2 sentiment rate: 100.0%


In [24]:
print(f"{'Metric':<25} {'v1 (dict)':>10} {'v2 (FinBERT)':>14}")
print("-" * 50)
print(f"{'Sentiment rate':<25} {V1_SENTIMENT_RATE:>9.1%} {v2['sentiment_present'].mean():>13.1%}")
print(f"{'Neutrality rate':<25} {1-V1_SENTIMENT_RATE:>9.1%} {1-v2['sentiment_present'].mean():>13.1%}")
improvement = (v2['sentiment_present'].mean() - V1_SENTIMENT_RATE) * 100

Metric                     v1 (dict)   v2 (FinBERT)
--------------------------------------------------
Sentiment rate                44.0%        100.0%
Neutrality rate               56.0%          0.0%


In [9]:
rows = []
for label, df in [("v1", v1), ("v2", v2)]:
    filtered = df[df["sentiment_present"]]
    daily_tickers = filtered.groupby(pd.Grouper(key="seendate", freq="D"))["ticker"].nunique()
    rows.append({
        "version": label,
        "articles_with_sentiment": filtered.shape[0],
        "sentiment_rate": f'{df["sentiment_present"].mean():.1%}',
        "days_5plus_tickers": (daily_tickers >= 5).sum(),
        "mean_tickers_per_day": daily_tickers.mean().round(2),
    })
pd.DataFrame(rows)

,version,articles_with_sentiment,sentiment_rate,days_5plus_tickers,mean_tickers_per_day
0,v1,3130,43.3%,33,0.33
1,v2,7224,100.0%,34,0.35


In [29]:
print(f"{'Threshold':<12} {'Sentiment Present':>18} {'Neutrality Rate':>16}")
print("-" * 48)
for threshold in [0.0, 0.05, 0.1, 0.15, 0.2, 0.25]:
    rate = (df['sentiment_score'].abs() > threshold).mean()
    print(f"> {threshold:<10.2f} {rate:>17.1%} {1-rate:>15.1%}")

Threshold     Sentiment Present  Neutrality Rate
------------------------------------------------
> 0.00                  100.0%            0.0%
> 0.05                   74.2%           25.8%
> 0.10                   62.8%           37.2%
> 0.15                   55.8%           44.2%
> 0.20                   51.5%           48.5%
> 0.25                   48.3%           51.7%


In [ ]:
THRESHOLD = 0.05 

df['sentiment_present'] = df['sentiment_score'].abs() > THRESHOLD

print(f"Threshold: {THRESHOLD}")
print(f"Sentiment present: {df['sentiment_present'].sum():,} / {len(df):,} ({df['sentiment_present'].mean():.1%})")
print()

# Sanity check — look at articles near the boundary
boundary = df[df['sentiment_score'].abs().between(0.0, 0.1)].sample(5, random_state=42)
print("Articles near the boundary:")
print(boundary[['title', 'sentiment_score', 'sentiment_label', 'sentiment_present']].to_string())

Threshold: 0.05
Sentiment present: 5,362 / 7,224 (74.2%)

Articles near the boundary:
                                                                                          title  sentiment_score sentiment_label  sentiment_present
3678                                           Match Group ( MTCH ) Q4 2025 Earnings Transcript         0.065131         neutral               True
5934  Linscomb Wealth Inc . Has $12 . 64 Million Stock Position in Meta Platforms , Inc . $META         0.019920         neutral              False
953                                Apple Card switches hands but no immediate changes for users        -0.033629         neutral              False
1384                      Serve Robotics ( NASDAQ : SERV ) CEO Ali Kashani Sells 9 , 088 Shares         0.034455         neutral              False
6907              OpenAI hires Meta AI researcher who previously led Apple models team : Report         0.028077         neutral              False
